# MCP Advanced Search Options

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Introduction</a>
* <a href="#setup">2 - Setup</a>
* <a href="#form-lemma">3 - Form search vs. lemma search</a>
* <a href="#operators">4 - Operator-preserving queries</a>
* <a href="#operator-notes">5 - Operator notes</a>
* <a href="#author-scope">6 - Author-scoped search</a>
* <a href="#guidance">7 - When to use each option</a>
* <a href="#sources">8 - Sources</a>
* <a href="#required-libraries">9 - Required libraries</a>
* <a href="#notebook-version">10 - Notebook version</a>

## 1 - Introduction <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook demonstrates the richer Scaife search options exposed by the `search_perseus` MCP tool: form search, lemma search, operator-preserving queries, and author-scoped filtering.

> Requirements: run from the repository root (or keep the path setup cell unchanged), install dependencies, and have internet access to Scaife/Perseus upstream services.

## 2 - Setup <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

Locate the repository root, import the local MCP server, and define compact helpers for displaying Scaife search responses.

In [ ]:
from pathlib import Path
import importlib
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from fastmcp import Client
import server

# Reload local edits when this notebook is rerun in an existing kernel.
server = importlib.reload(server)
mcp = server.mcp


def tool_text(result):
    return "\n".join(
        block.text for block in result.content if getattr(block, "text", None) is not None
    )


def summarize_search(data):
    results = data.get("results", [])
    first = results[0] if results else {}
    passage = first.get("passage", {})
    text = passage.get("text", {})
    snippet = " ".join(first.get("content", [])) if first else None
    return {
        "total_count": data.get("total_count"),
        "page": data.get("page", {}).get("number"),
        "num_pages": data.get("page", {}).get("num_pages"),
        "first_urn": passage.get("urn"),
        "first_text_label": text.get("label"),
        "first_snippet": snippet,
        "author_scope": data.get("author_scope"),
    }


def print_summary(label, data):
    print(f"\n## {label}")
    print(json.dumps(summarize_search(data), ensure_ascii=False, indent=2))


## 3 - Form search vs. lemma search <a class="anchor" id="form-lemma"></a>
##### [Back to ToC](#TOC)

`search_kind="form"` is the default and searches the surface form in the text. `search_kind="lemma"` asks Scaife to search by dictionary headword/lemma instead. Lemma search is useful when you want inflected forms grouped under the same lexical entry.

The examples below keep `query_format="unicode"` so the Unicode Greek query is sent as written.

In [ ]:
async with Client(mcp) as client:
    form_response = await client.call_tool(
        "search_perseus",
        {
            "query": "μῆνιν",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
        },
    )
    lemma_response = await client.call_tool(
        "search_perseus",
        {
            "query": "λόγος",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "lemma",
        },
    )

form_results = json.loads(tool_text(form_response))
lemma_results = json.loads(tool_text(lemma_response))

print_summary("form search: μῆνιν", form_results)
print_summary("lemma search: λόγος", lemma_results)



## form search: μῆνιν
{
  "total_count": 314,
  "page": 1,
  "num_pages": 32,
  "first_urn": "urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238",
  "first_text_label": "Διονυσιακά",
  "first_snippet": "<em>μῆνιν</em> ἀλυσκάζοντες ἀθηήτοιο Λυαίου",
  "author_scope": null
}

## lemma search: λόγος
{
  "total_count": 12052,
  "page": 1,
  "num_pages": 1206,
  "first_urn": "urn:cts:greekLit:tlg2200.tlg00459.opp-grc1:praef",
  "first_text_label": "Oratio 59",
  "first_snippet": "ΕΙΣ ΚΩΝΣΤΑΝΤΙΟΝ ΚΑΙ\n\nΚΩΝΣΤΑΝΤΑ. Προθεωρία τοῦ <em>λόγου</em>. R III 272 Ἐκεῖνο μάλιστα μὲν ἐκ μιᾶς ἀρχῆς ὁ <em>λόγος</em> καὶ\n\n C  codex Chisianus εἰς τοὺς αὐτοκράτορας κώνσταντα καὶ\n\nκωνστάντιον <em>λόγος</em> βασιλικὸς Ρ rubr εἰς τούς fol. 93 v λιβα-\n\nνίου σοφιστοῦ <em>λόγος</em> πρὸς κωνστάντιον καὶ κώνσταντα Ι τούς αὐτο-\n\nκράτορας Κώνσταντα καὶ Κωνστάντιον <em>λόγος</em> Mor 1 et Βασιλικὸς\n\npost <em>λόγος</em> <em>λόγος</em> Mor 1 et Βασιλικὸς\n\npost <em>λόγος</em> add Mor 2 εἰς τούς εἰς τούς αὐτ

## 4 - Operator-preserving queries <a class="anchor" id="operators"></a>
##### [Back to ToC](#TOC)

Scaife accepts several operator-like query patterns through the same `q` parameter. Use `preserve_operators=True` when your query includes syntax characters such as quotes, `-`, `|`, `*`, or `~`.

This matters because `query_format="auto"` treats some of these characters as Beta Code markers. For operator queries, prefer `query_format="unicode"` plus `preserve_operators=True` so the query reaches Scaife unchanged.

In [ ]:
operator_examples = [
    (
        "exact phrase",
        {
            "query": '"μῆνιν ἄειδε"',
            "language": "greek",
            "query_format": "unicode",
            "preserve_operators": True,
        },
    ),
    (
        "exclude term",
        {
            "query": "μῆνιν -ἄειδε",
            "language": "greek",
            "query_format": "unicode",
            "preserve_operators": True,
        },
    ),
    (
        "or-style query",
        {
            "query": "μῆνιν | ἄειδε",
            "language": "greek",
            "query_format": "unicode",
            "preserve_operators": True,
        },
    ),
    (
        "wildcard",
        {
            "query": "μῆν*",
            "language": "greek",
            "query_format": "unicode",
            "preserve_operators": True,
        },
    ),
    (
        "lemma or-style query",
        {
            "query": "λόγος | ἀνήρ",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "lemma",
            "preserve_operators": True,
        },
    ),
]

operator_results = {}
async with Client(mcp) as client:
    for label, arguments in operator_examples:
        response = await client.call_tool("search_perseus", arguments)
        operator_results[label] = json.loads(tool_text(response))

for label, data in operator_results.items():
    print_summary(label, data)



## exact phrase
{
  "total_count": 43,
  "page": 1,
  "num_pages": 5,
  "first_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "first_text_label": "Ἰλιάς",
  "first_snippet": "<em>μῆνιν ἄειδε</em> θεὰ Πηληϊάδεω Ἀχιλῆος",
  "author_scope": null
}

## exclude term
{
  "total_count": 262,
  "page": 1,
  "num_pages": 27,
  "first_urn": "urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238",
  "first_text_label": "Διονυσιακά",
  "first_snippet": "<em>μῆνιν</em> ἀλυσκάζοντες ἀθηήτοιο Λυαίου",
  "author_scope": null
}

## or-style query
{
  "total_count": 363,
  "page": 1,
  "num_pages": 37,
  "first_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "first_text_label": "Ἰλιάς",
  "first_snippet": "<em>μῆνιν</em> <em>ἄειδε</em> θεὰ Πηληϊάδεω Ἀχιλῆος",
  "author_scope": null
}

## wildcard
{
  "total_count": 20161,
  "page": 1,
  "num_pages": 2017,
  "first_urn": "urn:cts:greekLit:stoa0033a.tlg028.1st1K-grc1:4",
  "first_text_label": "De mundo",
  "first_snippet": "Περὶ δὲ

## 5 - Operator notes <a class="anchor" id="operator-notes"></a>
##### [Back to ToC](#TOC)

Live Scaife testing showed useful behavior for quoted phrases, exclusion with `-`, or-style queries with `|`, wildcard `*`, and fuzzy suffix `~`. Uppercase `AND` and `OR` did not behave like useful Boolean operators in testing, so this notebook uses the operator forms that changed result counts predictably.

The MCP tool does not parse or rewrite these operators. It preserves them and lets Scaife interpret the query.

## 6 - Author-scoped search <a class="anchor" id="author-scope"></a>
##### [Back to ToC](#TOC)

The optional `author` argument is applied after the Scaife request. The server resolves the author against CTS capabilities, then filters the current Scaife result page to hits whose CTS URNs fall under the matched author/work/resource URN prefixes.

This is local post-filtering over the returned page, not a server-side Scaife author parameter. The returned JSON includes an `author_scope` object so you can inspect what was matched and how many hits survived the page filter.

In [ ]:
async with Client(mcp) as client:
    scoped_response = await client.call_tool(
        "search_perseus",
        {
            "query": '"μῆνιν ἄειδε"',
            "language": "greek",
            "query_format": "unicode",
            "preserve_operators": True,
            "author": "Homer",
        },
    )

scoped_results = json.loads(tool_text(scoped_response))
print_summary('author-scoped exact phrase: Homer', scoped_results)



## author-scoped exact phrase: Homer
{
  "total_count": 43,
  "page": 1,
  "num_pages": 5,
  "first_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "first_text_label": "Ἰλιάς",
  "first_snippet": "<em>μῆνιν ἄειδε</em> θεὰ Πηληϊάδεω Ἀχιλῆος",
  "author_scope": {
    "query": "Homer",
    "match_count": 2,
    "urns": [
      "urn:cts:greekLit:tlg0012",
      "urn:cts:greekLit:tlg0012.tlg001",
      "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
      "urn:cts:greekLit:tlg0012.tlg001.perseus-eng1",
      "urn:cts:greekLit:tlg0012.tlg001.perseus-eng2",
      "urn:cts:greekLit:tlg0012.tlg002",
      "urn:cts:greekLit:tlg0012.tlg002.perseus-grc1",
      "urn:cts:greekLit:tlg0012.tlg002.perseus-eng1",
      "urn:cts:greekLit:tlg0012.tlg002.perseus-eng2",
      "urn:cts:greekLit:tlg0013",
      "urn:cts:greekLit:tlg0013.tlg027",
      "urn:cts:greekLit:tlg0013.tlg027.perseus-grc1",
      "urn:cts:greekLit:tlg0013.tlg027.perseus-eng1",
      "urn:cts:greekLit:tlg0013.tlg026",
   

## 7 - When to use each option <a class="anchor" id="guidance"></a>
##### [Back to ToC](#TOC)

- Use `search_kind="form"` when the exact surface form matters.
- Use `search_kind="lemma"` when you want lexical search across inflected forms.
- Use `preserve_operators=True` for quoted phrases, exclusion, or-style queries, wildcards, and fuzzy searches.
- Use `author` when you want a quick page-level scope filter after Scaife returns results.
- Use `find_author_names` or `get_author_resources` first when you need to verify which CTS author/textgroup will be used for author-scoped filtering.

## 8 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook draws on the following software and data services:

- the local Perseus MCP implementation in [`server.py`](../server.py) and the project documentation in the [README](../README.md);
- the [Perseus Digital Library](https://www.perseus.tufts.edu/) and its [CTS endpoint](https://www.perseus.tufts.edu/hopper/CTS) for text inventory, citation, and passage data;
- the [Scaife Viewer](https://scaife.perseus.org/) APIs for library search and Scaife-native metadata or passage retrieval;
- [FastMCP](https://github.com/jlowin/fastmcp) for the in-process MCP client/server interface.

The Perseus and Scaife services are live upstream sources. Available editions, result counts, ordering, and response details may change over time, so discovery cells should be rerun before reusing recorded URNs.

## 9 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

The repository requires **Python 3.11 or newer**. From the repository root, install the declared dependencies with:

```bash
pip install -e .
```

The principal third-party libraries used by this notebook are:

- `fastmcp>=2.12.0` for the MCP client and local server;
- `httpx>=0.27.0`, used by the server for upstream HTTP requests;
- Jupyter/IPython to run and display the notebook.

Modules such as `json`, `pathlib`, `re`, `sys`, and `xml.etree.ElementTree` are part of the Python standard library and do not need separate installation.

## 10 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.1</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>